# Phase 2 evaluation: outdated and contradicted facts in agent memory

Runs every experiment the Phase 2 brief asks for (E1 to E6), with the settings our supervisor asked for:
temperature 0, at least 3 seeds, confidence intervals, and LoCoMo only as a check that general recall is not lost.

| Experiment | What runs here | Data |
|---|---|---|
| E1 baseline | `rag` | FactConsolidation, change scenarios, LoCoMo |
| E2 base paper | `amem` (A-Mem's own code, reproduced by us) | same |
| E3 full system | `tsom` | same |
| E4 main gap | `tsom` vs `amem`: stale-answer rate, active contradiction rate | FactConsolidation, change scenarios |
| E5 ablation | `newest_wins`: the classifier removed | FactConsolidation, change scenarios |
| E6 efficiency | LLM calls, tokens, p50/p95 write latency for every system | all runs |

Also: extraction recall, failure categories with counts, success and failure traces, and the gap-resolution table.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Run the cells in order. Every result goes to Google Drive, and a
finished run is skipped when a cell runs again, so after a disconnect just run everything again from the top.

## 1. Settings
Change these before a run, and keep them the same for every result in the paper.

In [ ]:
import os, sys

COLAB = "google.colab" in sys.modules
BRANCH = "feat/gap-measurement"
COMMIT = ""            # freeze the code: put the commit hash used for the paper here ("" = newest on BRANCH)
MODEL = "qwen2.5:3b"
CONTEXT = 16384        # Ollama's default of 4096 silently cuts our prompts; every system gets this same limit

# Seeds per experiment. At temperature 0, Qwen gave identical replies for different seeds in our check,
# so the extra seeds of the 6K runs show that; the long runs use one seed. Report this choice in the paper.
SEEDS = {"fc6k": [1, 2, 3], "scenarios": [1, 2, 3], "fc32k": [1], "locomo": [1]}

# Experiments to run. The job list below runs them in order of importance.
RUN = {"extraction": True, "fc6k": True, "scenarios": True, "fc32k": True, "locomo": True,
       "fact_history": True,      # tsom with the agent's get_fact_history choice, on the scenarios
       "locomo_ablations": False} # newest_wins and the history tool on LoCoMo too: many more hours
# LoCoMo is slow: about an hour and a half per conversation for amem and tsom together. All 10 take days, so
# start with 2 (conversation 0, the development one, and conversation 1, a test one) and raise it if time allows.
# Finished conversations are kept, so raising it later only runs the new ones.
LOCOMO_CONVERSATIONS = 2

# Two people can split the jobs: each sets WORKERS = 2 and a different WORKER (0 or 1), runs the notebook on their
# own Colab, then copies their runs/ folder into one Drive folder before the analysis cells.
WORKERS, WORKER = 1, 0

# Small limits for a quick end-to-end check (None = everything)
MAX_QUESTIONS = None
SCENARIO_LIMIT = None

SYSTEMS = {               # name -> (config, extra overrides)
    "rag": ("configs/rag_qwen2.5-3b.json", []),
    "amem": ("configs/amem_qwen2.5-3b.json", []),
    "tsom": ("configs/tsom_qwen2.5-3b.json", []),
    "newest_wins": ("configs/tsom_newest_qwen2.5-3b.json", []),
}
BACKEND = "ollama" if COLAB else os.environ.get("EVAL_BACKEND", "fake")  # outside Colab: a fake model, for checking the notebook

## 2. Code and data
Results go to Drive: `MyDrive/g1_results/phase2/`.

In [ ]:
import subprocess, json, time
from pathlib import Path

if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/g1_results/phase2")
    REPO = Path("/content/repo")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/FatimaSohailll/tsom-agent-memory.git", str(REPO)], check=True)
    if COMMIT:
        subprocess.run(["git", "-C", str(REPO), "checkout", "-q", COMMIT], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    OUT = Path(os.environ.get("EVAL_OUT", "eval_out"))
    REPO = Path.cwd()

sys.path.insert(0, str(REPO))  # so the analysis cells can import the repo's code
RUNS, LOGS = OUT / "runs", OUT / "logs"
for folder in (RUNS, LOGS):
    folder.mkdir(parents=True, exist_ok=True)
CODE_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_COMMIT, "| results:", OUT)

## 3. Model
Starts Ollama with the larger context limit and checks that it took effect.

In [ ]:
import urllib.request

def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if BACKEND == "ollama":
    if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    subprocess.run(["pkill", "ollama"])
    time.sleep(2)
    subprocess.Popen(["ollama", "serve"], env=dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(CONTEXT)),
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
    subprocess.run(["ollama", "pull", MODEL], check=True, capture_output=True)
    body = json.dumps({"model": MODEL, "prompt": "Reply with one word: READY", "stream": False}).encode()
    urllib.request.urlopen(urllib.request.Request("http://localhost:11434/api/generate", data=body), timeout=300)
    running = json.load(urllib.request.urlopen("http://localhost:11434/api/ps"))["models"]
    table = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
    print(table)
    context = running[0].get("context_length")
    if context is None:  # older Ollama: read the CONTEXT column of `ollama ps`
        header, row = table.splitlines()[:2]
        context = int(row[header.index("CONTEXT"):].split()[0])
    assert context == CONTEXT, f"Ollama is using a context of {context}, not {CONTEXT}: results would be cut"
else:
    print("backend:", BACKEND, "(no model; results are meaningless, this only checks the notebook)")

## 4. Record the setup
Saved as `setup.json` next to the results, for the paper's experimental setup section.

In [ ]:
def version(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return None

setup = {
    "code_commit": CODE_COMMIT, "branch": BRANCH, "backend": BACKEND, "model": MODEL,
    "model_id": version(["ollama", "list"]) if BACKEND == "ollama" else None,
    "ollama": version(["ollama", "--version"]) if BACKEND == "ollama" else None,
    "context_length": CONTEXT, "temperature": 0.0, "seeds": SEEDS, "locomo_conversations": LOCOMO_CONVERSATIONS,
    "systems": SYSTEMS, "python": sys.version.split()[0],
    "gpu": version(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]),
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
}
(OUT / "setup.json").write_text(json.dumps(setup, indent=2))
print(json.dumps(setup, indent=2))

## 5. Tests
Offline tests: they must pass before any run counts.

In [ ]:
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1])
assert tests.returncode == 0, tests.stdout[-3000:]

## 6. Job list
Every run, in order of importance, with a rough time on a T4. The times are estimates from one 6K `tsom` run
(about 14 minutes to store 455 facts) and the number of LLM calls each system makes; check them against the logs.

| Data | Size used | Why |
|---|---|---|
| FactConsolidation SH and MH | 6K and 32K, 100 questions each | the main benchmark, named in the proposal |
| Change scenarios | every reviewed scenario | personal facts; added values, which FactConsolidation lacks |
| LoCoMo | the first `LOCOMO_CONVERSATIONS` conversations | the no-harm check only |

In [ ]:
DONE_FILES = {"factcons": "summary_factcons_{variant}_{size}.json", "scenarios": "scenarios_summary.json",
              "locomo": "summary.json", "extraction": "extraction_summary.json"}
FC_MINUTES = {"rag": 5, "amem": 10, "newest_wins": 13, "tsom": 19}          # per 6K run, roughly
LOCOMO_TURN_SECONDS = {"rag": 0, "amem": 6, "tsom": 8, "newest_wins": 7}   # per stored turn, roughly
LOCOMO_TURNS = [419, 369, 663, 629, 680, 675, 689, 681, 509, 568]
LOCOMO_QUESTIONS = [199, 105, 193, 260, 242, 158, 190, 239, 196, 204]

JOBS = []

def job(group, kind, system, seed, name, minutes, args=(), overrides=(), **fill):
    JOBS.append(dict(group=group, kind=kind, system=system, seed=seed, name=name, minutes=minutes,
                     args=list(args), overrides=list(overrides), fill=fill))

def fc_args(variant, size):
    args = ["--variant", variant, "--size", size, "--max-samples", "1"]
    return args + (["--max-questions", str(MAX_QUESTIONS)] if MAX_QUESTIONS else [])

def fc_jobs(group, size, seeds, factor):
    for seed in seeds:
        for variant in ("sh", "mh"):
            for system in SYSTEMS:
                job(group, "factcons", system, seed, f"fc_{variant}{size}_{system}_s{seed}",
                    FC_MINUTES[system] * factor, fc_args(variant, size), variant=variant, size=size)

def scenario_jobs(seeds):
    args = ["--limit", str(SCENARIO_LIMIT)] if SCENARIO_LIMIT else []
    for seed in seeds:
        for system in SYSTEMS:
            job("scenarios", "scenarios", system, seed, f"scen_{system}_s{seed}", 2, args)
        if RUN["fact_history"]:
            job("scenarios", "scenarios", "tsom_history", seed, f"scen_tsom_history_s{seed}", 2, args,
                ["use_fact_history=true"])

def locomo_jobs(systems):
    turns, questions = sum(LOCOMO_TURNS[:LOCOMO_CONVERSATIONS]), sum(LOCOMO_QUESTIONS[:LOCOMO_CONVERSATIONS])
    for seed in SEEDS["locomo"]:
        for system in systems:
            config_system = "tsom" if system == "tsom_history" else system
            minutes = (turns * LOCOMO_TURN_SECONDS[config_system] + questions * 3) / 60
            extra = ["use_fact_history=true"] if system == "tsom_history" else []
            job("locomo", "locomo", system, seed, f"locomo_{system}_s{seed}", minutes,
                overrides=[f"max_samples={LOCOMO_CONVERSATIONS}", *extra])

first, rest = SEEDS["fc6k"][:1], SEEDS["fc6k"][1:]
if RUN["extraction"]:
    job("extraction", "extraction", "extraction", 1, "extraction", 2,
        ["--limit", str(SCENARIO_LIMIT)] if SCENARIO_LIMIT else [])
if RUN["fc6k"]:
    fc_jobs("fc6k", "6k", first, 1)
if RUN["scenarios"]:
    scenario_jobs(SEEDS["scenarios"][:1])
if RUN["fc32k"]:
    fc_jobs("fc32k", "32k", SEEDS["fc32k"][:1], 5)
if RUN["locomo"]:
    locomo_jobs(["rag", "amem", "tsom"])
if RUN["fc6k"]:
    fc_jobs("fc6k", "6k", rest, 1)
if RUN["scenarios"]:
    scenario_jobs(SEEDS["scenarios"][1:])
if RUN["fc32k"]:
    fc_jobs("fc32k", "32k", SEEDS["fc32k"][1:], 5)
if RUN["locomo"] and RUN["locomo_ablations"]:
    locomo_jobs(["newest_wins"] + (["tsom_history"] if RUN["fact_history"] else []))

MINE = [j for i, j in enumerate(JOBS) if i % WORKERS == WORKER]
done = lambda j: (RUNS / j["name"] / DONE_FILES[j["kind"]].format(**j["fill"])).exists()
left = [j for j in MINE if not done(j)]
print(f"{len(MINE)} jobs for this worker, {len(left)} not finished, about {sum(j['minutes'] for j in left) / 60:.1f} hours left")
for j in MINE:
    print(f"  {'done ' if done(j) else '     '} {j['name']:32} ~{j['minutes']:.0f} min")

## 7. How a run is started
Every run gets its own folder under `runs/` and a log under `logs/`. A finished run is skipped.

In [ ]:
MODULES = {"factcons": "harness.run_factcons", "scenarios": "harness.run_scenarios",
           "locomo": "harness.run_locomo", "extraction": "harness.eval_extraction"}

def command(j):
    config, extra = SYSTEMS.get(j["system"], SYSTEMS["tsom"])
    sets = [f"run_name={j['name']}", f"output_dir={RUNS}", "temperature=0", f"seed={j['seed']}", "send_seed=true",
            *extra, *j["overrides"]]
    if BACKEND != "ollama":
        sets += [f"backend={BACKEND}", f"model={BACKEND}"]
    return [sys.executable, "-m", MODULES[j["kind"]], "--config", config, *j["args"], "--set", *sets]

def run(j):
    if done(j):
        return
    cmd = command(j)
    print(f"running {j['name']} (~{j['minutes']:.0f} min)", flush=True)
    start = time.time()
    with open(LOGS / f"{j['name']}.log", "a") as log:
        result = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)
    print(f"  {'finished' if result.returncode == 0 else 'FAILED'} in {(time.time() - start) / 60:.1f} min", flush=True)
    if result.returncode != 0:
        print((LOGS / f"{j['name']}.log").read_text()[-3000:])
        raise RuntimeError(f"{j['name']} failed; see logs/{j['name']}.log")

## 8. Run every job
The long cell. If Colab disconnects, run every cell again from the top: finished runs are skipped, and LoCoMo
keeps each finished conversation. To stop early, interrupt this cell and run the analysis cells below on what is done.

In [ ]:
for j in MINE:
    run(j)
print("all jobs for this worker finished")

## 9. Collect the run folders
Reads every run folder present under `runs/`, including ones copied over from the other worker.

In [ ]:
FOLDERS = {}
for j in JOBS:
    folder = RUNS / j["name"]
    if folder.exists():
        FOLDERS.setdefault(j["system"], []).append(folder)
for system, folders in FOLDERS.items():
    print(f"{system:14} {len(folders)} run folders")

## 10. Main tables with confidence intervals
Mean across seeds, the spread (sd) when there are several seeds, and a bootstrap 95% CI over questions or scenarios.

In [ ]:
FOLDERS = {s: sorted(set(f for f in fs if f.exists())) for s, fs in FOLDERS.items()}
cmd = [sys.executable, "-m", "harness.make_tables", "--out", str(OUT / "tables.md")]
for system, folders in FOLDERS.items():
    if system == "tsom":
        folders = folders + FOLDERS.get("extraction", [])  # extraction recall is shown as the fact layer's row
    if folders and system != "extraction":
        cmd += ["--run", f"{system}=" + ",".join(map(str, folders))]
subprocess.run(cmd, check=True, capture_output=True)
print((OUT / "tables.md").read_text(encoding="utf-8"))

## 11. LoCoMo by conversation split
`make_tables` uses every conversation that ran. This splits off conversation 0, the development conversation.

In [ ]:
from harness.metrics import summarise

def locomo_rows(folder, test_only):
    rows = []
    for path in sorted(folder.glob("sample_*.jsonl")):
        rows += [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    return [r for r in rows if not test_only or r["sample"] != 0]

lines = ["| Run | Conversations | " + " | ".join(["Single-hop", "Multi-hop", "Temporal", "Open-domain", "Adversarial", "Overall"])
         + " |", "|---" * 8 + "|"]
for system, folders in FOLDERS.items():
    for folder in folders:
        if folder.name.startswith("locomo_"):
            for label, test_only in (("all", False), ("test only (no conversation 0)", True)):
                scores = summarise(locomo_rows(folder, test_only))
                cells = [f"{scores[c]['f1']:.1f} (n={scores[c]['n']})" if c in scores else "-"
                         for c in ("Single-hop", "Multi-hop", "Temporal", "Open-domain", "Adversarial", "Overall")]
                lines.append(f"| {folder.name} | {label} | " + " | ".join(cells) + " |")
(OUT / "locomo_split.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines) if len(lines) > 2 else "No LoCoMo runs yet.")

## 12. Efficiency and reliability (E6)
Per memory write (A-Mem's write and our fact step together): LLM calls, tokens and latency, with p50 and p95.
Reliability: fact steps that failed, and classifier replies that could not be read (`UNRESOLVED`).

In [ ]:
from statistics import median

def percentile(values, p):
    values = sorted(values)
    return values[min(len(values) - 1, int(round(p / 100 * (len(values) - 1))))] if values else float("nan")

def efficiency(folder):
    per_write, unresolved, failures, answers = {}, 0, 0, []
    for path in folder.glob("ops_sample_*.jsonl"):
        for op in map(json.loads, path.read_text(encoding="utf-8").splitlines()):
            if op["op_type"] in ("write", "temporal_write"):
                key = (path.name, op["ids_touched"][0] if op["ids_touched"] else id(op))
                w = per_write.setdefault(key, {"calls": 0, "tokens": 0, "seconds": 0.0})
                w["calls"] += op["llm_calls"]
                w["tokens"] += op["prompt_tokens"] + op["completion_tokens"]
                w["seconds"] += op["latency_s"]
                unresolved += op.get("op_counts", {}).get("UNRESOLVED", 0)
                failures += op.get("failures", 0)
            elif op["op_type"] == "answer":
                answers.append(op["prompt_tokens"] + op["completion_tokens"])
    writes = list(per_write.values())
    if not writes:
        return None
    seconds = [w["seconds"] for w in writes]
    return {"writes": len(writes), "calls_per_write": sum(w["calls"] for w in writes) / len(writes),
            "tokens_per_write": sum(w["tokens"] for w in writes) / len(writes),
            "p50_s": median(seconds), "p95_s": percentile(seconds, 95),
            "tokens_per_answer": sum(answers) / len(answers) if answers else float("nan"),
            "unresolved": unresolved, "failed_fact_steps": failures}

lines = ["| Run | Writes | LLM calls/write | Tokens/write | p50 s/write | p95 s/write | Tokens/answer | UNRESOLVED | Failed fact steps |",
         "|---|---|---|---|---|---|---|---|---|"]
for system, folders in FOLDERS.items():
    for folder in folders:
        e = efficiency(folder)
        if e:
            lines.append(f"| {folder.name} | {e['writes']} | {e['calls_per_write']:.2f} | {e['tokens_per_write']:.0f} | "
                         f"{e['p50_s']:.2f} | {e['p95_s']:.2f} | {e['tokens_per_answer']:.0f} | {e['unresolved']} | "
                         f"{e['failed_fact_steps']} |")
(OUT / "efficiency.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## 13. Failure analysis
Every wrong FactConsolidation answer of the fact-layer systems, put in one category by checking the fact store
and the memory the answer was given:

| Category | Meaning | Likely cause |
|---|---|---|
| stale, correct value was shown | the newest value was `[CURRENT]` in the memory given to the model, yet it answered an older value | the model trusted its own knowledge or an old note |
| stale, correct value not shown | the newest value was current in the store but missing from that memory | retrieval or the fact cap |
| stale, store kept the old value | the older value was still current in the store | a wrong classifier decision |
| newest value never stored | no stored fact holds the right answer | extraction missed it |
| wrong, not stale | some other wrong answer | the answer step |

In [ ]:
from collections import Counter
from tsom.store import Store

def has(facts, values, status=None):
    values = [v.lower() for v in values if v]
    return any(v in f.value.lower() for f in facts for v in values if status is None or f.status == status)

def shown_current(row):
    lines = [l.lower() for l in row.get("context", "").splitlines() if l.startswith("[CURRENT]")]
    return any(g.lower() in l for l in lines for g in row["gold"] if g)

def categorise(row, facts):
    if not has(facts, row["gold"]):
        return "newest value never stored"
    if row.get("stale"):
        if has(facts, row["gold"], "active"):
            if "context" not in row:
                return "stale, store was right (context not saved)"
            return "stale, correct value was shown" if shown_current(row) else "stale, correct value not shown"
        if has(facts, row["older"], "active"):
            return "stale, store kept the old value"
    return "wrong, not stale"

def store_facts(folder, tag):
    db = folder / f"memory_sample_{tag}.sqlite"
    if not db.exists():
        return None
    store = Store(str(db))
    facts = [f for key in store.entity_keys(tag) for f in store.entity_facts(tag, key)]
    store.close()
    return facts

failures, totals, examples = Counter(), Counter(), {}
for system in ("tsom", "newest_wins"):
    for folder in FOLDERS.get(system, []):
        for path in folder.glob("sample_*_*.jsonl"):
            tag = path.stem[len("sample_"):]
            facts = store_facts(folder, tag)
            if facts is None:
                continue
            for row in map(json.loads, path.read_text(encoding="utf-8").splitlines()):
                totals[system] += 1
                if not row["correct"]:
                    category = categorise(row, facts)
                    failures[(system, category)] += 1
                    examples.setdefault((system, category), (folder.name, row))

lines = ["| System | Failure category | Count | Share of all questions |", "|---|---|---|---|"]
lines += [f"| {s} | {c} | {n} | {100 * n / totals[s]:.1f}% |" for (s, c), n in sorted(failures.items())]
(OUT / "failures.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
for (system, category), (run_name, row) in sorted(examples.items()):
    print(f"\n[{system}] {category} ({run_name})\n  Q: {row['question']}\n"
          f"  answered: {row['prediction']} | gold: {row['gold']} | older: {row['older']}")

## 14. Success and failure traces (agent trajectories)
Two successes and two failures each from FactConsolidation and the change scenarios, for `tsom`. Each trace shows
the agent's steps: what was stored and how the fact layer changed, the tool calls at question time, the memory the
model was given, and its answer. The brief asks for at least two of each in the paper.

In [ ]:
def entity_history(facts, names):
    names = [n.lower() for n in names]
    keep = [f for f in facts if any(n and n in f.entity.lower() for n in names)]
    return "\n".join(f"  step {f.observed_at:>4}  {f.status:10} {f.entity} | {f.attribute} | {f.value}  ({f.operation})"
                     for f in sorted(keep, key=lambda f: f.observed_at)) or "  (no stored facts about it)"

def memory_excerpt(context, limit=2500):
    return context if len(context) <= limit else context[:limit] + f"\n  ... ({len(context) - limit} more characters)"

traces = []
fc = [(folder, path) for folder in FOLDERS.get("tsom", []) for path in sorted(folder.glob("sample_sh_*.jsonl"))]
if fc:
    folder, path = fc[0]
    tag = path.stem[len("sample_"):]
    facts = store_facts(folder, tag) or []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
    picks = [("SUCCESS", r) for r in rows if r["correct"] and r.get("older")][:2]
    picks += [("FAILURE", r) for r in rows if r.get("stale")][:2]
    for label, r in picks:
        entities = [f.entity for f in facts if f.entity.lower() in r["question"].lower()]
        traces.append(
            f"=== FactConsolidation {label}: {folder.name} ===\n"
            f"Question: {r['question']}\nGold: {r['gold']}  Older values: {r['older']}\n"
            f"1. Writes: the fact layer's record for {sorted(set(entities)) or 'the question'}\n"
            f"{entity_history(facts, entities)}\n"
            f"2. Tool calls: {json.dumps(r.get('tools', []))}\n"
            f"3. Memory given to the model:\n{memory_excerpt(r.get('context', '(not saved in this run)'))}\n"
            f"4. Answer: {r['prediction']}  -> {'correct' if r['correct'] else 'stale' if r['stale'] else 'wrong'}\n")

scen = []
for folder in FOLDERS.get("tsom", []):
    path = folder / "scenarios_results.jsonl"
    if path.exists():
        scen = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
        break
picks = [("SUCCESS", r) for r in scen if r["correct"] and not r["hedged"] and r["kind"] != "restate"][:2]
picks += [("FAILURE", r) for r in scen if r["stale"] or not r["correct"]][:2]
for label, r in picks:
    a = r["answers"][0]
    facts = "\n".join(f"  step {f['step']:>4}  {f['status']:10} {f['entity']} | {f['attribute']} | {f['value']}  ({f['operation']})"
                      for f in r.get("facts", [])) or (
        "  (no facts: this system stores none)" if "facts" in r else "  (not saved in this run)")
    traces.append(
        f"=== Change scenario {label}: {r['id']} ({r['kind']}) ===\n"
        f"Question: {a['question']}\nExpected: {r['current']}  Old: {r['old']}\n"
        f"1. Writes: the fact layer at the end of the conversation\n{facts}\n"
        f"2. Tool calls: {json.dumps(a.get('tools', []))}\n"
        f"3. Memory given to the model:\n{memory_excerpt(a['context'])}\n"
        f"4. Answer: {a['prediction']}  -> {a['verdict']}\n")

(OUT / "traces.md").write_text("\n".join(traces), encoding="utf-8")
print("\n".join(traces) if traces else "No results yet.")

## 15. Do seeds change anything?
At temperature 0 the seed should not matter. This counts how many answers are identical between seeds, which is
the evidence for running the long experiments with one seed.

In [ ]:
from collections import defaultdict

def answers(folder):
    out = {}
    for path in sorted(folder.glob("sample_*.jsonl")):
        for i, row in enumerate(map(json.loads, path.read_text(encoding="utf-8").splitlines())):
            out[(path.name, i)] = row["prediction"]
    path = folder / "scenarios_results.jsonl"
    if path.exists():
        for row in map(json.loads, path.read_text(encoding="utf-8").splitlines()):
            out[(row["id"], 0)] = row["answers"][0]["prediction"]
    return out

groups = defaultdict(dict)
for j in JOBS:
    folder = RUNS / j["name"]
    if folder.exists():
        groups[j["name"].rsplit("_s", 1)[0]][j["seed"]] = answers(folder)
lines = ["| Experiment | Seeds | Answers compared | Identical across seeds |", "|---|---|---|---|"]
for name, by_seed in sorted(groups.items()):
    if len(by_seed) < 2:
        continue
    keys = set.intersection(*(set(a) for a in by_seed.values()))
    same = sum(len({a[k] for a in by_seed.values()}) == 1 for k in keys)
    lines.append(f"| {name} | {sorted(by_seed)} | {len(keys)} | {same} ({100 * same / max(len(keys), 1):.1f}%) |")
(OUT / "seeds.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## 16. Change scenarios by kind
Where the systems differ: `change` and `cancel` test retiring old values, `additive` tests keeping both values,
`restate` tests not losing a value that was said again. Seed 1 only. Cells are k/n.

In [ ]:
from harness.scenarios import fmt

lines = ["| System | Kind | Scenarios | Correct | Stale | Hedged | Retention |", "|---|---|---|---|---|---|---|"]
for system, folders in FOLDERS.items():
    for folder in folders:
        path = folder / "scenarios_summary.json"
        if not path.exists() or not folder.name.endswith("_s1"):
            continue
        summary = json.loads(path.read_text(encoding="utf-8"))
        for kind, b in summary["per_kind"].items():
            lines.append(f"| {system} | {kind} | {b['scenarios']} | {fmt(b['correct'])} | {fmt(b['stale'])} | "
                         f"{fmt(b['hedged'])} | {fmt(b['retention'])} |")
(OUT / "scenarios_by_kind.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

## 17. Figures for the paper
Saved as PNG and PDF in `figures/`. Error bars are bootstrap 95% confidence intervals over questions or scenarios.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from harness.make_tables import bootstrap_ci

FIGS = OUT / "figures"
FIGS.mkdir(exist_ok=True)
ORDER = [s for s in ("rag", "amem", "newest_wins", "tsom", "tsom_history") if s in FOLDERS]

def save(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=200)
    plt.close(fig)
    print("saved", name)

def fc_outcomes(system, tag, key):
    for folder in FOLDERS.get(system, []):
        path = folder / f"sample_{tag}.jsonl"
        if path.exists() and folder.name.endswith("_s1"):
            rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
            return [bool(r[key]) for r in rows if key != "stale" or r.get("older")]
    return []

def bars(ax, labels, outcomes, title):
    means = [100 * sum(o) / len(o) if o else 0 for o in outcomes]
    cis = [bootstrap_ci(o) or (m, m) for o, m in zip(outcomes, means)]
    err = [[m - lo for m, (lo, hi) in zip(means, cis)], [hi - m for m, (lo, hi) in zip(means, cis)]]
    ax.bar(labels, means, yerr=err, capsize=4, color="#4C72B0")
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("%")
    ax.set_ylim(0, 100)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

tags = sorted({p.stem[len("sample_"):] for fs in FOLDERS.values() for f in fs for p in f.glob("sample_[sm]h_*.jsonl")})
if tags:
    fig, axes = plt.subplots(2, len(tags), figsize=(3.2 * len(tags), 6), squeeze=False)
    for col, tag in enumerate(tags):
        for row, (key, name) in enumerate((("correct", "Accuracy"), ("stale", "Stale answers"))):
            shown = [s for s in ORDER if fc_outcomes(s, tag, key)]
            bars(axes[row][col], shown, [fc_outcomes(s, tag, key) for s in shown], f"{name}, {tag}")
    save(fig, "factconsolidation")

curves = []
for folder in FOLDERS.get("tsom", []):
    path = folder / "summary_factcons_sh_6k.json"
    if path.exists() and folder.name.endswith("_s1"):
        block = json.loads(path.read_text(encoding="utf-8")).get("contradictions")
        if block:
            curves = list(block["per_sample"].values())[0]
        break
if curves:
    fig, ax = plt.subplots(figsize=(5, 3.2))
    steps = [p["step"] for p in curves]
    ax.plot(steps, [p["rate"] for p in curves], marker="o", label="tsom: active facts")
    ax.plot(steps, [p["rate_without_consolidation"] for p in curves], marker="s", label="nothing retired (A-Mem)")
    ax.set_xlabel("fact number (checkpoint)")
    ax.set_ylabel("contradictions per 100 facts")
    ax.set_title("Active contradictions, FactConsolidation SH 6K", fontsize=10)
    ax.legend(fontsize=8)
    save(fig, "contradictions")

kinds = ["change", "cancel", "additive", "restate"]
scen = {}
for system in ORDER:
    for folder in FOLDERS.get(system, []):
        path = folder / "scenarios_results.jsonl"
        if path.exists() and folder.name.endswith("_s1"):
            scen[system] = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
if scen:
    fig, axes = plt.subplots(1, len(kinds), figsize=(3.2 * len(kinds), 3.2), squeeze=False)
    for col, kind in enumerate(kinds):
        bars(axes[0][col], list(scen), [[r["correct"] for r in scen[s] if r["kind"] == kind] for s in scen],
             f"Correct, {kind}")
    save(fig, "scenarios_by_kind")

cost = [(s, efficiency(f)) for s in ORDER for f in FOLDERS.get(s, []) if f.name.startswith("fc_sh6k_") and f.name.endswith("_s1")]
cost = [(s, e) for s, e in cost if e]
if cost:
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
    axes[0].bar([s for s, _ in cost], [e["calls_per_write"] for _, e in cost], color="#55A868")
    axes[0].set_title("LLM calls per write (SH 6K)", fontsize=10)
    x = range(len(cost))
    axes[1].bar([i - 0.2 for i in x], [e["p50_s"] for _, e in cost], width=0.4, label="p50")
    axes[1].bar([i + 0.2 for i in x], [e["p95_s"] for _, e in cost], width=0.4, label="p95")
    axes[1].set_xticks(list(x), [s for s, _ in cost])
    axes[1].set_title("Seconds per write (SH 6K)", fontsize=10)
    axes[1].legend(fontsize=8)
    for ax in axes:
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    save(fig, "efficiency")

## 18. Comparison with the A-Mem paper
The brief asks whether base-paper numbers were reproduced. Ours were reproduced with A-Mem's own code, at
temperature 0 and with our 16K context, on the conversations listed. Copy the published Qwen2.5-3B row from
A-Mem's Table 1 into `PAPER` (we do not fill it in for you, so nothing is mistyped).

In [ ]:
PAPER = {}  # e.g. {"Single-hop": 0.0, "Multi-hop": 0.0, "Temporal": 0.0, "Open-domain": 0.0, "Adversarial": 0.0}

lines = ["| Category | A-Mem paper (Table 1, Qwen2.5-3B) | A-Mem, our reproduction (all conversations run) | Test conversations only |",
         "|---|---|---|---|"]
for folder in FOLDERS.get("amem", []):
    if folder.name.startswith("locomo_"):
        everything, test = summarise(locomo_rows(folder, False)), summarise(locomo_rows(folder, True))
        for cat in everything:
            paper = PAPER.get(cat)
            lines.append(f"| {cat} | {paper if paper is not None else 'copy from paper'} | "
                         f"{everything[cat]['f1']:.1f} | {test[cat]['f1']:.1f} |" if cat in test else
                         f"| {cat} | {paper if paper is not None else 'copy from paper'} | {everything[cat]['f1']:.1f} | - |")
        break
(OUT / "base_paper.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines) if len(lines) > 2 else "No A-Mem LoCoMo run yet.")

## 19. Gap-resolution table (draft)
Numbers come straight from the runs above. The last column is your judgement, so it is left for you to fill in.

In [ ]:
from statistics import mean

def fc_rate(system, tag, key):
    values = []
    for folder in FOLDERS.get(system, []):
        path = folder / f"sample_{tag}.jsonl"
        if path.exists():
            rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]
            pool = [r for r in rows if r.get("older")] if key == "stale" else rows
            if pool:
                values.append(100 * mean(bool(r[key]) for r in pool))
    return f"{mean(values):.1f}%" if values else "not run"

def contradiction(system, tag):
    found = [json.loads((f / f"summary_factcons_{tag}.json").read_text()) for f in FOLDERS.get(system, [])
             if (f / f"summary_factcons_{tag}.json").exists()]
    blocks = [x["contradictions"] for x in found if x.get("contradictions")]
    if not blocks:
        return "not run"
    return (f"{mean(b['final_rate'] for b in blocks):.2f} vs "
            f"{mean(b['final_rate_without_consolidation'] for b in blocks):.2f} with nothing retired")

def calls_per_write(system):
    folders = [f for f in FOLDERS.get(system, []) if f.name.startswith("fc_sh6k_")]
    values = [e["calls_per_write"] for e in map(efficiency, folders) if e]
    return f"{mean(values):.2f}" if values else "not run"

rows = [("Outdated and contradicted facts (Gap 1 + 3)", "Fact layer: classifier + validity", "Stale-answer rate, FactConsolidation SH 6K",
         fc_rate("amem", "sh_6k", "stale"), fc_rate("tsom", "sh_6k", "stale")),
        ("", "", "Stale-answer rate, FactConsolidation MH 6K (estimate)", fc_rate("amem", "mh_6k", "stale"), fc_rate("tsom", "mh_6k", "stale")),
        ("", "", "Accuracy, FactConsolidation SH 6K", fc_rate("amem", "sh_6k", "correct"), fc_rate("tsom", "sh_6k", "correct")),
        ("", "", "Active contradictions per 100 facts, SH 6K", "no facts stored", contradiction("tsom", "sh_6k")),
        ("Write cost (Gap 2, measured only)", "none (A-Mem's evolution unchanged)", "LLM calls per write, FactConsolidation SH 6K",
         calls_per_write("amem"), calls_per_write("tsom"))]
lines = ["| Gap | Method / component | Metric | Base paper result (A-Mem, reproduced) | Our result | Addressed? |",
         "|---|---|---|---|---|---|"] + [f"| {' | '.join(r)} | fill in |" for r in rows]
(OUT / "gap_table.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## 20. Experiment index and commands
Which run folders make up each experiment, and the exact command that reproduces each run. Goes into the
repository as `experiments.md`, so the README can point to it.

In [ ]:
PURPOSE = {
    "rag": "E1 baseline", "amem": "E2 base paper (A-Mem, reproduced)", "tsom": "E3 full system; E4 main gap",
    "newest_wins": "E5 ablation: classifier removed", "tsom_history": "E3 variant: agent chooses get_fact_history",
    "extraction": "E4 extraction recall"}
lines = ["# Experiments", "", f"Code commit `{CODE_COMMIT}`, model `{MODEL}`, temperature 0, context {CONTEXT}.",
         "E6 (cost, latency, reliability) is computed from every run's `ops_sample_*.jsonl`.", "",
         "| Experiment | Run | Command |", "|---|---|---|"]
for j in JOBS:
    if (RUNS / j["name"]).exists():
        shown = " ".join(command(j)[1:]).replace(str(RUNS), "results/phase2/runs")
        lines.append(f"| {PURPOSE.get(j['system'], j['system'])} | `{j['name']}` | `python {shown}` |")
(OUT / "experiments.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

## 21. Files for the repository
Copies what can be committed into `for_repo/`: summaries, tables, figures, the setup record, traces and short
excerpts of the per-question logs. Full `sample_*.jsonl`, `ops_sample_*.jsonl` and `memory_sample_*.sqlite` files
stay on Drive and are not committed.

In [ ]:
import shutil

KEEP = ("summary.json", "summary.md", "scenarios_summary.json", "scenarios_summary.md", "extraction_summary.json",
        "extraction_summary.md", "config.json")
target = OUT / "for_repo"
for folder in sorted(RUNS.iterdir()):
    for path in folder.iterdir():
        if path.name in KEEP or path.name.startswith("summary_factcons_"):
            (target / "runs" / folder.name).mkdir(parents=True, exist_ok=True)
            shutil.copy(path, target / "runs" / folder.name / path.name)
for name in ("setup.json", "tables.md", "efficiency.md", "failures.md", "gap_table.md", "traces.md", "seeds.md",
             "scenarios_by_kind.md", "locomo_split.md", "base_paper.md", "experiments.md"):
    if (OUT / name).exists():
        shutil.copy(OUT / name, target / name)
if (OUT / "figures").exists():
    shutil.copytree(OUT / "figures", target / "figures", dirs_exist_ok=True)
excerpts = target / "trace_excerpts"
excerpts.mkdir(exist_ok=True)
for folder in sorted(RUNS.iterdir()):
    for path in list(folder.glob("sample_*.jsonl"))[:1]:
        (excerpts / f"{folder.name}.jsonl").write_text("\n".join(path.read_text(encoding="utf-8").splitlines()[:3]) + "\n",
                                                       encoding="utf-8")
print("Copy this folder into the repo's results/phase2/:", target)